# IFCNetCore — DINOv3 ViT-L/16 visual embeddings

Runs DINOv3 ViT-L/16 (`facebookresearch/dinov3`) over all 12 stock renders per
object at 768×768, mean-pools the CLS token across views, writes one 1024-d
`.npy` per object.

**Upload to your Drive:** `IFCNetCorePng.zip` (445 MB renders zip).
**Output to Drive:** `dinov3_ifcnet_colorless.zip`.

Resume-safe: re-running skips obj_ids whose `.npy` already exists.

⚠️ The weights filename must match the hash pattern `*lvd1689m-*.pth` —
do NOT rename to `model.pth` (hubconf validates the filename).

Use a fresh runtime to avoid conflicts with SigLIP / DuoDuo notebooks.
Pick a GPU runtime first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
from pathlib import Path

DRIVE_ZIP    = Path('/content/drive/MyDrive/IFCNetCorePng.zip')
DRIVE_OUT    = Path('/content/drive/MyDrive')

RENDERS_ROOT  = Path('/content/data/IFCNetCore/renders')
FEATURES_ROOT = Path('/content/data/IFCNetCore/processed/rendered_features')
METADATA_PATH = Path('/content/data/IFCNetCore/metadata.json')

assert DRIVE_ZIP.exists(), f'Upload IFCNetCorePng.zip to {DRIVE_ZIP.parent} first.'
print('zip   :', DRIVE_ZIP, '(', DRIVE_ZIP.stat().st_size // (1024*1024), 'MB)')
print('drive :', DRIVE_OUT)

In [ ]:
RENDERS_ROOT.mkdir(parents=True, exist_ok=True)
!unzip -q -n "$DRIVE_ZIP" -d "$RENDERS_ROOT"
n_png = sum(1 for _ in RENDERS_ROOT.rglob('*.png'))
print(f'unzipped: {n_png} PNGs')

In [ ]:
import json
rows = []
for class_dir in sorted(RENDERS_ROOT.iterdir()):
    if not class_dir.is_dir(): continue
    for split_dir in sorted(class_dir.iterdir()):
        if not split_dir.is_dir(): continue
        obj_ids = sorted({p.stem.rsplit('.', 1)[0] for p in split_dir.glob('*.png')})
        for oid in obj_ids:
            views = sorted(split_dir.glob(f'{oid}.*.png'))
            rows.append({'obj_id': oid, 'ifc_class': class_dir.name,
                         'split': split_dir.name, 'num_renders': len(views)})
METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
METADATA_PATH.write_text(json.dumps(rows, indent=2))
print(f'{len(rows)} objects, {len({r["ifc_class"] for r in rows})} classes')

In [ ]:
import json, shutil, subprocess
from dataclasses import dataclass
from tqdm import tqdm
import numpy as np


@dataclass(frozen=True)
class ObjectEntry:
    obj_id: str
    ifc_class: str
    split: str
    views: list


def load_entries():
    out = []
    for r in json.loads(METADATA_PATH.read_text()):
        d = RENDERS_ROOT / r['ifc_class'] / r['split']
        out.append(ObjectEntry(r['obj_id'], r['ifc_class'], r['split'],
                               sorted(d.glob(f"{r['obj_id']}.*.png"))))
    return out


def filter_pending(entries, out_dir):
    return [e for e in entries if not (out_dir / f'{e.obj_id}.npy').exists()]


def zip_and_upload(features_subdir, archive_name, drive_out=DRIVE_OUT):
    src = features_subdir / 'colorless'
    n = sum(1 for _ in src.glob('*.npy'))
    if n == 0:
        print(f'  [warn] {src} empty — nothing to zip'); return None
    local = Path('/content') / archive_name
    if local.exists(): local.unlink()
    subprocess.run(['zip', '-qr', str(local), 'colorless'], cwd=str(features_subdir), check=True)
    drive_out.mkdir(parents=True, exist_ok=True)
    drive_zip = drive_out / archive_name
    shutil.copy2(local, drive_zip)
    print(f'  ✓ zipped {n} files → {drive_zip}  ({drive_zip.stat().st_size/1024/1024:.1f} MB)')
    return drive_zip


ENTRIES = load_entries()
print(f'{len(ENTRIES)} object entries')

## Clone repo + download weights

In [ ]:
if not Path('/content/dinov3').exists():
    !git clone -q https://github.com/facebookresearch/dinov3.git /content/dinov3
!ls /content/dinov3 | head

In [ ]:
# Filename MUST match the *lvd1689m-*.pth hash pattern (hubconf validates).
DINOV3_WEIGHTS = Path('/content/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth')
if not DINOV3_WEIGHTS.exists():
    !wget -q -O "$DINOV3_WEIGHTS" https://huggingface.co/jaychempan/dinov3/resolve/44126792d766b593994f73c1019d7788a2a715e6/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
print(f'weights: {DINOV3_WEIGHTS.stat().st_size // (1024*1024)} MB')

## Run DINOv3 (~3.5 h on T4 for 7930 objects)

In [ ]:
import torch
import torchvision.transforms.functional as TF
from PIL import Image

PATCH_SIZE = 16
IMAGE_SIZE = 768
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DINOV3_OUT = FEATURES_ROOT / 'dinov3' / 'colorless'
DINOV3_OUT.mkdir(parents=True, exist_ok=True)


def resize_for_dinov3(pil):
    w, h = pil.size
    h_patches = int(IMAGE_SIZE / PATCH_SIZE)
    w_patches = int((w * IMAGE_SIZE) / (h * PATCH_SIZE))
    resized = TF.resize(pil, (h_patches * PATCH_SIZE, w_patches * PATCH_SIZE))
    tens = TF.to_tensor(resized)
    return TF.normalize(tens, mean=IMAGENET_MEAN, std=IMAGENET_STD)


pending = filter_pending(ENTRIES, DINOV3_OUT)
print(f'[dinov3] {len(pending)} / {len(ENTRIES)} pending')

if pending:
    dinov3 = torch.hub.load('/content/dinov3', 'dinov3_vitl16',
                            source='local',
                            weights=str(DINOV3_WEIGHTS),
                            skip_validation=True)
    dinov3 = dinov3.cuda().eval()

    # smoke
    with torch.no_grad():
        _t = resize_for_dinov3(Image.open(pending[0].views[0]).convert('RGB')).unsqueeze(0).cuda()
        _out = dinov3(_t)
    print(f'[dinov3] smoke: out type={type(_out).__name__}  shape={tuple(_out.shape)}')

    for e in tqdm(pending, desc='dinov3'):
        try:
            tensors = [resize_for_dinov3(Image.open(v).convert('RGB')).unsqueeze(0)
                       for v in e.views]
            batch = torch.cat(tensors).cuda()
            with torch.no_grad():
                feats = dinov3(batch)
            vec = feats.mean(dim=0).cpu().numpy().astype(np.float32)
            np.save(DINOV3_OUT / f'{e.obj_id}.npy', vec)
        except Exception as exc:
            print(f'  [error] {e.obj_id}: {exc}')

    del dinov3
    torch.cuda.empty_cache()

print('done:', len(list(DINOV3_OUT.glob('*.npy'))), 'embeddings written')

In [ ]:
zip_and_upload(FEATURES_ROOT / 'dinov3',
               archive_name='dinov3_ifcnet_colorless.zip')